# Data collection: SpaceX API

Isse Abdallah Mohamoud. IBM Applied Data Science Capstone historical snapshots. Run from the spacex-capstone folder.

The archived API response preserves the original course scope. Download only when the local archive is missing; normalize and execute the exact filter-count audit below. Related-entity enrichment is supplied by the prepared IBM CSV.

In [1]:
from pathlib import Path
import json, requests, pandas as pd
from IPython.display import display
R=Path('.')
archive_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
p=R/'data/API_call_spacex_api.json'
if not p.exists():
    response=requests.get(archive_url,timeout=60)
    response.raise_for_status()
    p.write_bytes(response.content)
raw=pd.json_normalize(json.loads(p.read_text()))
print('Raw API rows:',len(raw))
print('First static-fire timestamp:',raw.static_fire_date_utc.iloc[0])
display(raw[['name','date_utc','rocket']].head())

Raw API rows: 107
First static-fire timestamp: 2006-03-17T00:00:00.000Z


,name,date_utc,rocket
0,FalconSat,2006-03-24T22:30:00.000Z,5e9d0d95eda69955f709d1eb
1,DemoSat,2007-03-21T01:10:00.000Z,5e9d0d95eda69955f709d1eb
2,Trailblazer,2008-08-03T03:34:00.000Z,5e9d0d95eda69955f709d1eb
3,RatSat,2008-09-28T23:15:00.000Z,5e9d0d95eda69955f709d1eb
4,RazakSat,2009-07-13T03:35:00.000Z,5e9d0d95eda69955f709d1eb


In [2]:
single=raw[(raw.cores.map(len)==1)&(raw.payloads.map(len)==1)].copy()
cutoff=single[pd.to_datetime(single.date_utc,utc=True)<='2020-11-13T23:59:59Z']
falcon9=cutoff[cutoff.rocket=='5e9d0d95eda69973a809d1ec']
audit=pd.DataFrame({'stage':['raw','single core and payload','date cutoff','Falcon 9 only'],'records':[len(raw),len(single),len(cutoff),len(falcon9)]})
display(audit)
assert audit.records.tolist()==[107,95,94,90]
prepared=pd.read_csv(R/'data/dataset_part_1.csv')
assert len(prepared)==len(falcon9)==90
display(prepared.head())

,stage,records
0,raw,107
1,single core and payload,95
2,date cutoff,94
3,Falcon 9 only,90


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
